In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np
import os


In [2]:
dataset_handle = "sergionefedov/language-learning-to-fluency-who-gets-there"

# Download the latest dataset
dataset_path = kagglehub.dataset_download(dataset_handle)
print("Dataset downloaded to:", dataset_path)

# List files included in the dataset
import os

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        print(os.path.join(root, file))


Dataset downloaded to: C:\Users\nicol\.cache\kagglehub\datasets\sergionefedov\language-learning-to-fluency-who-gets-there\versions\1
C:\Users\nicol\.cache\kagglehub\datasets\sergionefedov\language-learning-to-fluency-who-gets-there\versions\1\data_dictionary_language.csv
C:\Users\nicol\.cache\kagglehub\datasets\sergionefedov\language-learning-to-fluency-who-gets-there\versions\1\language_learning_fluency.csv


In [3]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

dic = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "sergionefedov/language-learning-to-fluency-who-gets-there",
    "data_dictionary_language.csv",
)
display(dic)

,column,type,description
0,learner_id,string,Unique learner id
1,fsi_category,int,Target-language difficulty (FSI 1=easy like Sp...
2,related_language,int,1 if the learner knows a language related to t...
3,prior_languages,int,Number of languages already known
4,age_started,int,Age when the learner started (affects accent c...
5,total_study_hours,int,Total study hours (the biggest single lever)
6,active_use_share,float,QUALITY: fraction of study that is active use/...
7,immersion_months,float,QUALITY: months living/using the language imme...
8,speaking_practice,float,QUALITY: speaking/output practice (0-10)
9,feedback_quality,float,"QUALITY: corrective feedback quality, e.g. a t..."


In [4]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "sergionefedov/language-learning-to-fluency-who-gets-there",
    "language_learning_fluency.csv",
)

print("First 5 records:")
print(df.head())


First 5 records:
  learner_id  fsi_category  related_language  prior_languages  age_started  \
0  LNR000000             2                 0                0           14   
1  LNR000001             1                 1                1           28   
2  LNR000002             2                 0                0           14   
3  LNR000003             4                 0                0           33   
4  LNR000004             2                 0                0           35   

   total_study_hours  active_use_share  immersion_months  speaking_practice  \
0                590             0.276               0.0               0.61   
1                960             0.294               0.0               4.50   
2                270             0.132               0.0               8.53   
3                 70             0.266               6.5               8.52   
4                780             0.746               0.0               5.80   

   feedback_quality  comprehensible_inp

In [5]:
print(df.info)

<bound method DataFrame.info of       learner_id  fsi_category  related_language  prior_languages  \
0      LNR000000             2                 0                0   
1      LNR000001             1                 1                1   
2      LNR000002             2                 0                0   
3      LNR000003             4                 0                0   
4      LNR000004             2                 0                0   
...          ...           ...               ...              ...   
44995  LNR044995             4                 0                2   
44996  LNR044996             1                 0                1   
44997  LNR044997             1                 0                1   
44998  LNR044998             2                 0                0   
44999  LNR044999             1                 1                0   

       age_started  total_study_hours  active_use_share  immersion_months  \
0               14                590             0.276       

In [15]:
null_count = df.isnull().sum()
print(f'Null values in dataset; {null_count}')

Null values in dataset; learner_id                    0
fsi_category                  0
related_language              0
prior_languages               0
age_started                   0
total_study_hours             0
active_use_share              0
immersion_months              0
speaking_practice             0
feedback_quality              0
comprehensible_input_hours    0
uses_srs                      0
motivation                    0
consistency                   0
accent_nativelike             0
cefr_level                    0
reached_fluency               0
dropped_out                   0
dtype: int64


In [6]:
display(df.head())

,learner_id,fsi_category,related_language,prior_languages,age_started,total_study_hours,active_use_share,immersion_months,speaking_practice,feedback_quality,comprehensible_input_hours,uses_srs,motivation,consistency,accent_nativelike,cefr_level,reached_fluency,dropped_out
0,LNR000000,2,0,0,14,590,0.276,0.0,0.61,1.27,430,0,4.96,7.29,6.66,A2,0,0
1,LNR000001,1,1,1,28,960,0.294,0.0,4.50,7.63,600,0,4.20,6.31,3.90,C1,1,0
2,LNR000002,2,0,0,14,270,0.132,0.0,8.53,0.00,120,1,1.93,6.80,7.57,A1,0,1
3,LNR000003,4,0,0,33,70,0.266,6.5,8.52,5.59,40,0,6.60,9.17,3.84,A0,0,0
4,LNR000004,2,0,0,35,780,0.746,0.0,5.80,4.74,470,1,1.90,6.48,5.97,C1,1,0


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
 

#Setting value to be predicted (target) and traits on which to base it (features)
target = df['cefr_level']
display(target)

features = df.drop(columns= ['learner_id','reached_fluency','cefr_level'])



display(features)

0        A2
1        C1
2        A1
3        A0
4        C1
         ..
44995    A1
44996    B1
44997    C2
44998    B2
44999    A2
Name: cefr_level, Length: 45000, dtype: object

,fsi_category,related_language,prior_languages,age_started,total_study_hours,active_use_share,immersion_months,speaking_practice,feedback_quality,comprehensible_input_hours,uses_srs,motivation,consistency,accent_nativelike,dropped_out
0,2,0,0,14,590,0.276,0.0,0.61,1.27,430,0,4.96,7.29,6.66,0
1,1,1,1,28,960,0.294,0.0,4.50,7.63,600,0,4.20,6.31,3.90,0
2,2,0,0,14,270,0.132,0.0,8.53,0.00,120,1,1.93,6.80,7.57,1
3,4,0,0,33,70,0.266,6.5,8.52,5.59,40,0,6.60,9.17,3.84,0
4,2,0,0,35,780,0.746,0.0,5.80,4.74,470,1,1.90,6.48,5.97,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44995,4,0,2,26,450,0.258,0.9,6.14,4.88,300,0,3.71,2.85,8.74,1
44996,1,0,1,35,360,0.582,0.0,1.93,2.56,240,0,7.60,9.15,6.04,0
44997,1,0,1,23,650,0.525,0.6,9.43,5.62,240,0,5.36,7.39,4.53,0
44998,2,0,0,31,460,0.885,0.0,10.00,6.63,370,0,5.57,7.89,3.20,0


In [8]:
#Normalizing values
features_train, features_test, target_train, target_test = train_test_split(features, target, test_size=0.2, random_state=42,
    stratify=target)
numeric_cols = features_train.select_dtypes(include="number").columns
scaler = MinMaxScaler()


features_train[numeric_cols] = scaler.fit_transform(features_train[numeric_cols])
features_test[numeric_cols] = scaler.transform(features_test[numeric_cols])

In [9]:
model = RandomForestClassifier(random_state=42)
model.fit(features_train, target_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [10]:
log_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=2000,
        random_state=42
    )
)

log_model.fit(features_train, target_train)


,steps,"[('standardscaler', ...), ('logisticregression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0


In [11]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=7) #Seven language proficiency levels
)

knn_model.fit(features_train, target_train)


,steps,"[('standardscaler', ...), ('kneighborsclassifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,n_neighbors,7
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30


In [12]:
from sklearn.ensemble import HistGradientBoostingClassifier

hist_model = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    random_state=42
)

hist_model.fit(features_train, target_train)


,loss,'log_loss'
,learning_rate,0.05
,max_iter,300
,max_leaf_nodes,31
,max_depth,None
,min_samples_leaf,20
,l2_regularization,0.0
,max_features,1.0
,max_bins,255
,categorical_features,'from_dtype'
,monotonic_cst,None


In [13]:
from sklearn.ensemble import ExtraTreesClassifier

extra_model = ExtraTreesClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

extra_model.fit(features_train, target_train)


,n_estimators,300
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,False
,oob_score,False


In [14]:
#Calculating how far off the false predictions are

cefr_order = {
    "A0": 0,
    "A1": 1,
    "A2": 2,
    "B1": 3,
    "B2": 4,
    "C1": 5,
    "C2": 6
}

true_numeric = target_test.map(cefr_order)
pred_numeric = pd.Series(target_pred, index=target_test.index).map(cefr_order)

mean_level_error = (true_numeric - pred_numeric).abs().mean()

print(f"Mean CEFR-level error: {mean_level_error:.2f}")

NameError: name 'target_pred' is not defined

In [ ]:
#Testing multiple models
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


print("------randomforestclassifier-------------")
target_pred = model.predict(features_test)

print("Accuracy:", round(accuracy_score(target_test, target_pred),3))
print("\nClassification report:")
print(classification_report(target_test, target_pred))

pred_numeric = pd.Series(target_pred, index=target_test.index).map(cefr_order)
mean_level_error = (true_numeric - pred_numeric).abs().mean()
print(f"Mean CEFR-level error: {mean_level_error:.2f}")


print("\nConfusion matrix:")
print(confusion_matrix(target_test, target_pred))

print("------logistic regression-------------")
log_target_pred = log_model.predict(features_test) 


print("Accuracy:", round(accuracy_score(target_test, log_target_pred),3))
print("\nClassification report:")
print(classification_report(target_test, log_target_pred))

log_pred_numeric = pd.Series(log_target_pred, index=target_test.index).map(cefr_order)
log_mean_level_error = (true_numeric - log_pred_numeric).abs().mean()
print(f"Mean CEFR-level error: {log_mean_level_error:.2f}")

print("\nConfusion matrix:")
print(confusion_matrix(target_test, log_target_pred))

print("------KNN-------------")
knn_target_pred = knn_model.predict(features_test) 

print("Accuracy:", round(accuracy_score(target_test, knn_target_pred),3))
print("\nClassification report:")
print(classification_report(target_test, knn_target_pred))

knn_pred_numeric = pd.Series(knn_target_pred, index=target_test.index).map(cefr_order)
knn_mean_level_error = (true_numeric - knn_pred_numeric).abs().mean()
print(f"Mean CEFR-level error: {knn_mean_level_error:.2f}")

print("\nConfusion matrix:")
print(confusion_matrix(target_test, knn_target_pred))

print("------gradient boosting-------------")
hist_target_pred = hist_model.predict(features_test) 

print("Accuracy:", round(accuracy_score(target_test, hist_target_pred),3))
print("\nClassification report:")
print(classification_report(target_test, hist_target_pred))

hist_pred_numeric = pd.Series(hist_target_pred, index=target_test.index).map(cefr_order)
hist_mean_level_error = (true_numeric - pred_numeric).abs().mean()
print(f"Mean CEFR-level error: {hist_mean_level_error:.2f}")

print("\nConfusion matrix:")
print(confusion_matrix(target_test, hist_target_pred))

print("------extra trees-------------")
extra_target_pred = extra_model.predict(features_test) 

print("Accuracy:", round(accuracy_score(target_test, extra_target_pred),3))
print("\nClassification report:")
print(classification_report(target_test, extra_target_pred))

extra_pred_numeric = pd.Series(extra_target_pred, index=target_test.index).map(cefr_order)
extra_mean_level_error = (true_numeric - extra_pred_numeric).abs().mean()
print(f"Mean CEFR-level error: {extra_mean_level_error:.2f}")

print("\nConfusion matrix:")
print(confusion_matrix(target_test, extra_target_pred))

#Investigating which features were most influential in each model

In [ ]:
importance = permutation_importance(model, features_test, target_test, n_repeats=10, random_state=42)

importance_df = pd.DataFrame({
    "feature": features_test.columns,
    "importance_mean": importance.importances_mean,
    "importance_std": importance.importances_std
}).sort_values("importance_mean", ascending=False)

display(importance_df)


In [ ]:
log_importance = permutation_importance(log_model, features_test, target_test, n_repeats=10, random_state=42)

log_importance_df = pd.DataFrame({
    "feature": features_test.columns,
    "importance_mean": log_importance.importances_mean,
    "importance_std": log_importance.importances_std
}).sort_values("importance_mean", ascending=False)

display(log_importance_df)

In [ ]:
knn_importance = permutation_importance(knn_model, features_test, target_test, n_repeats=10, random_state=42)

knn_importance_df = pd.DataFrame({
    "feature": features_test.columns,
    "importance_mean": knn_importance.importances_mean,
    "importance_std": knn_importance.importances_std
}).sort_values("importance_mean", ascending=False)

display(knn_importance_df)

In [ ]:
hist_importance = permutation_importance(hist_model, features_test, target_test, n_repeats=10, random_state=42)

hist_importance_df = pd.DataFrame({
    "feature": features_test.columns,
    "importance_mean": hist_importance.importances_mean,
    "importance_std": hist_importance.importances_std
}).sort_values("importance_mean", ascending=False)

display(hist_importance_df)

In [ ]:
extra_importance = permutation_importance(extra_model, features_test, target_test, n_repeats=10, random_state=42)

extra_importance_df = pd.DataFrame({
    "feature": features_test.columns,
    "importance_mean": extra_importance.importances_mean,
    "importance_std": extra_importance.importances_std
}).sort_values("importance_mean", ascending=False)

display(extra_importance_df)